<a href="https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/00_START_HERE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[⬅ Back to the course index](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/00_START_HERE.ipynb) · **Notebook 00 of the course**


# 🩺 START HERE — Practical Machine Learning for Medical Research

**Welcome!** This is a hands-on day: you will clean, explore, model and interrogate **real ICU
time-series data** from sepsis patients, and leave with notebooks you can point at *your own*
clinical CSV tomorrow.

This notebook does three things — takes about **5 minutes**:

1. checks your environment is ready,
2. loads the workshop data **once** and caches it, so every other notebook just works,
3. gives you the clickable index of the whole course.

> **New to Python?** That's fine. You never have to write code from scratch today — you run cells,
> read what happened, and edit the bits marked **✏️ Your turn**.


## 📦 Step 1 — the data (nothing to do)

The two workshop files are **not** in the public GitHub repo: they are real de-identified patient
records, covered by a data use agreement, so they cannot be redistributed openly. They live behind a
download link instead — and **that link is already built into these notebooks**, so the data arrives
by itself when you run the next two cells.

| File | Size | What it is |
|------|-----:|------------|
| `sepsis_timeseries.csv` | ≈9 MB | the main file — one row per patient per 4-hour block |
| `sepsis_patients.csv` | ≈1 MB | one row per patient (notebooks 05–08, 13, 14 can rebuild it if missing) |

It takes a few seconds the first time. After that it is cached — in Colab, to your Google Drive — so
every other notebook, and every future session, opens it instantly.

> 📎 **If the download ever fails**, the same link is posted in the **course Teams channel**: paste it
> into `WORKSHOP_DATA_URL` at the top of the next cell. Failing that, the notebook will offer to let
> you upload the two CSVs by hand.

**Running locally instead?** Put both CSVs in a `data/` folder next to the notebooks and they are
found with no download at all.

## ⚙️ Step 2 — run the setup cell

It installs anything missing and defines the data loader. It is **identical in every notebook** of
the course, so you'll see it again — just run it and move on.

In [ ]:
# === ⚙️  Workshop setup — run this cell first ===============================
# Works in Google Colab and in local Jupyter. Installs anything missing, sets a
# clean plotting style, and gives you helpers to load the data.
# (This cell is identical in every notebook of the course.)
import importlib.util, subprocess, sys, os, random, warnings
warnings.filterwarnings("ignore")

# --- reproducibility: everyone in the room gets the same numbers ---------------
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)

# 📥 Where the workshop data comes from — already set up for you, nothing to do.
# The data downloads automatically the first time you need it. If you were given a
# different link, just paste it in place of the one below. These forms all work:
#   • a Google-Drive folder link      • a Drive / Dropbox / OneDrive file link
#   • a folder URL ending in "/"      • a link straight to a .zip
# Set it to "" if you would rather upload the CSVs by hand.
# (The data is not in the GitHub repo: it is real de-identified patient data covered
#  by a data use agreement and may not be redistributed openly.)
WORKSHOP_DATA_URL = os.environ.get(
    "WORKSHOP_DATA_URL",
    "https://drive.google.com/drive/folders/1y7CparhrqdlCAZq6xQlj8fZda394fniD")

def _ensure(pkgs):
    missing = [pip for mod, pip in pkgs.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing])
_ensure({"numpy":"numpy","pandas":"pandas","sklearn":"scikit-learn",
         "matplotlib":"matplotlib","seaborn":"seaborn","shap":"shap","xgboost":"xgboost"})

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(RANDOM_STATE)          # seeds the legacy global np.random.* calls
RNG = np.random.default_rng(RANDOM_STATE)   # the modern generator — use this one
pd.set_option("display.max_columns", 120); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5); plt.rcParams["figure.dpi"] = 110

# Every model, split and resample in this course passes random_state=RANDOM_STATE, so
# your numbers should match your neighbour's exactly. (Different library *versions* can
# still shift the last decimal — that is normal and not a mistake on your part.)

# --- data loading: works locally AND remembers your upload across notebooks -----
_CACHE = {"dir": "unset"}   # memo so we only touch Google Drive once per session

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _drive_cache():
    """In Google Colab, mount Drive ONCE and return a persistent folder. A file you
    upload in one notebook is saved here, so every other notebook opens it automatically
    — no re-uploading. Returns None outside Colab, or if you decline to connect Drive."""
    if _CACHE["dir"] != "unset":
        return _CACHE["dir"]
    result = None
    if _in_colab():
        try:
            from google.colab import drive
            if not os.path.ismount("/content/drive"):
                drive.mount("/content/drive")
            result = "/content/drive/MyDrive/sepsis_workshop_data"
            os.makedirs(result, exist_ok=True)
        except Exception:
            result = None
    _CACHE["dir"] = result
    return result

def _find(name):
    """Look for the file on disk. Deliberately does NOT touch Google Drive, so the normal
    path never triggers an authorisation popup."""
    for p in [name, f"data/{name}", f"../data/{name}", f"workshop/data/{name}"]:
        if os.path.exists(p):
            return p
    return None

def _find_in_drive(name):
    """Only used as a fallback, because it mounts Drive (and that means a popup)."""
    cache = _drive_cache()
    if cache:
        p = os.path.join(cache, name)
        if os.path.exists(p):
            return p
    return None

def _direct_url(u):
    """Turn an ordinary Google-Drive / Dropbox / OneDrive *share* link into one that a
    plain HTTP client can actually download, so you can paste the link you were given."""
    import re
    m = (re.search(r"drive\.google\.com/file/d/([\w-]+)", u)
         or re.search(r"drive\.google\.com/(?:open|uc)\?(?:export=\w+&)?id=([\w-]+)", u))
    if m:
        return f"https://drive.google.com/uc?export=download&id={m.group(1)}"
    if "dropbox.com" in u:
        return u.split("?")[0] + "?dl=1"
    if "sharepoint.com" in u or "1drv.ms" in u:
        return u + ("&" if "?" in u else "?") + "download=1"
    return u

def _fetch(url, dest):
    import urllib.request, shutil as _sh
    req = urllib.request.Request(_direct_url(url), headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
        _sh.copyfileobj(r, f)

_FOLDER = {"done": False}

def _gdrive_folder(name):
    """WORKSHOP_DATA_URL points at a Google-Drive *folder*: fetch it once with gdown
    (a folder cannot be downloaded with a plain HTTP request), then serve files from it."""
    dest = "_workshop_data"
    if not _FOLDER["done"]:
        _ensure({"gdown": "gdown"})
        import gdown
        print("⬇  fetching the workshop data from Google Drive (just once) …")
        gdown.download_folder(url=WORKSHOP_DATA_URL, output=dest, quiet=True, use_cookies=False)
        _FOLDER["done"] = True
    for root, _dirs, files in os.walk(dest):
        if name in files:
            return os.path.join(root, name)
    return None

def _try_download(name):
    """Fetch the data from WORKSHOP_DATA_URL, if one was configured."""
    u = (WORKSHOP_DATA_URL or "").strip()
    if not u:
        return None
    try:
        if "/drive/folders/" in u:
            return _gdrive_folder(name)
        if u.lower().split("?")[0].endswith(".zip"):
            import zipfile
            bundle = "_workshop_data.zip"
            if not os.path.exists(bundle):
                print("⬇  downloading the workshop data bundle …")
                _fetch(u, bundle)
            with zipfile.ZipFile(bundle) as z:      # flatten any folder inside the zip
                for member in z.namelist():
                    if os.path.basename(member) == name:
                        with z.open(member) as src, open(name, "wb") as dst:
                            dst.write(src.read())
                        return name
            print(f"  ({name} was not inside the bundle)")
            return None
        print(f"⬇  downloading {name} …")
        _fetch(u.rstrip("/") + "/" + name, name)
        return name
    except Exception as e:
        print(f"  (download failed: {e})")
        for leftover in (name, "_workshop_data.zip"):
            if os.path.exists(leftover) and os.path.getsize(leftover) == 0:
                os.remove(leftover)
        return None

def _cache_to_drive(name, data):
    cache = _drive_cache()
    if cache:
        dest = os.path.join(cache, name)
        data.to_csv(dest, index=False)
        print(f"  💾 saved to Google Drive ({dest}) — no need to fetch it again.")

def load_csv(name):
    """Load a data CSV: looks on disk, then downloads it from WORKSHOP_DATA_URL, then checks
    your Google-Drive cache, and only as a last resort asks you to upload it — in which case
    it saves a copy to Drive so you never have to upload it twice."""
    p = _find(name)                       # 1. already on disk?
    if p:
        print(f"✓ loaded {p}")
        return pd.read_csv(p)
    p = _try_download(name)               # 2. the built-in download link
    if p:
        print(f"✓ loaded {name}")
        return pd.read_csv(p)
    p = _find_in_drive(name)              # 3. a copy you saved on a previous run
    if p:
        print(f"✓ loaded {p}")
        return pd.read_csv(p)
    try:                                  # 4. last resort: upload it by hand
        from google.colab import files
        print(f"⤴  Upload {name} just once — I'll save it so the other notebooks open it automatically:")
        up = files.upload()
        fname = list(up.keys())[0]
        data = pd.read_csv(fname)
        _cache_to_drive(name, data)
        return data
    except Exception:
        raise FileNotFoundError(
            f"Could not find {name}. Either paste your download link into WORKSHOP_DATA_URL at "
            f"the top of this cell, or put the CSV next to this notebook / in a data/ folder."
        )


## ✅ Step 3 — load the data once and check everything works

Run this. If you see a wall of green ticks, you are ready for the day.

In [2]:
# --- environment + data self-check -------------------------------------------
import importlib, platform

print(f"Python {platform.python_version()}   ({'Google Colab' if _in_colab() else 'local Jupyter'})\n")
for mod, minimum in [("pandas", "2.0"), ("numpy", "1.24"), ("sklearn", "1.3"),
                     ("matplotlib", "3.7"), ("seaborn", "0.12"), ("xgboost", "1.7"), ("shap", "0.42")]:
    try:
        v = importlib.import_module(mod).__version__
        print(f"  ✅ {mod:<12} {v}")
    except Exception as e:
        print(f"  ❌ {mod:<12} MISSING  ({e})")

print("\nLoading the workshop data (this is the one-and-only upload) …\n")
ts = load_csv("sepsis_timeseries.csv")
print(f"  ✅ time-series : {ts.shape[0]:,} rows × {ts.shape[1]} columns, "
      f"{ts['icustayid'].nunique():,} ICU stays")

try:
    pat = load_csv("sepsis_patients.csv")
    print(f"  ✅ patient table: {pat.shape[0]:,} patients × {pat.shape[1]} columns")
except FileNotFoundError:
    pat = None
    print("  ⚠️  sepsis_patients.csv not loaded — that's fine, notebooks 05+ rebuild it automatically.")

print(f"\n  90-day mortality in this cohort: {ts.groupby('icustayid')['morta_90'].max().mean():.1%}")
print("\n🎉 You're ready. Open Notebook 01 from the index below.")

Python 3.12.10   (local Jupyter)

  ✅ pandas       3.0.3
  ✅ numpy        2.4.6
  ✅ sklearn      1.9.0
  ✅ matplotlib   3.11.1
  ✅ seaborn      0.13.2
  ✅ xgboost      3.3.0


  ✅ shap         0.52.0

Loading the workshop data (this is the one-and-only upload) …

✓ loaded data/sepsis_timeseries.csv
  ✅ time-series : 37,704 rows × 53 columns, 1,696 ICU stays
✓ loaded data/sepsis_patients.csv
  ✅ patient table: 1,696 patients × 107 columns

  90-day mortality in this cohort: 18.2%

🎉 You're ready. Open Notebook 01 from the index below.


### 👀 A first look at the data

One row = **one patient at one 4-hour block**. Hold on to that sentence; the whole day rests on it.

In [3]:
ts.loc[ts["icustayid"] == ts["icustayid"].iloc[0],
       ["icustayid", "bloc", "HR", "SysBP", "Arterial_lactate", "SOFA", "morta_90"]].head(8)

,icustayid,bloc,HR,SysBP,Arterial_lactate,SOFA,morta_90
0,30003226,1,83.40,141.75,2.00,1,0
1,30003226,2,69.71,147.86,1.10,7,0
2,30003226,3,66.25,169.50,0.60,6,0
3,30003226,4,72.12,145.50,1.00,6,0
4,30003226,5,75.25,134.00,0.80,4,0
5,30003226,6,64.43,144.25,0.70,4,0
6,30003226,7,75.00,114.75,0.97,6,0
7,30003226,8,69.20,114.75,0.97,6,0


---

## 🗺️ The course index — click any notebook to open it in Colab

| # | Notebook | You'll learn | ~min |
|---|----------|--------------|-----:|
| 01 | [Pandas for clinical data](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/01_pandas_for_clinical_data.ipynb) | load, filter, groupby, dates, merge | 55 |
| 02 | [Cleaning messy clinical data](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/02_data_cleaning.ipynb) | missingness, unit errors, outliers, duplicates | 55 |
| 03 | [Exploratory data analysis](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/03_exploratory_data_analysis.ipynb) | distributions, correlations, trajectories | 35 |
| 04 | [⭐ Timeline → features](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/04_time_series_feature_engineering.ipynb) | lags, deltas, rolling windows, leakage-safe aggregation | 70 |
| 05 | [Your first predictive models](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/05_first_ml_models.ipynb) | pipelines, split by patient, LogReg → RF → XGBoost | 65 |
| 06 | [Evaluating like a clinician](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/06_model_evaluation.ipynb) | ROC, PR, thresholds, calibration | 45 |
| 07 | [Opening the black box (SHAP)](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/07_explainable_ai.ipynb) | global + per-patient explanations | 35 |
| 08 | [How to fool yourself](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/08_pitfalls_and_leakage.ipynb) | leakage, patient overlap, temporal validation | 40 |
| 09 | [Optimizing your model](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/09_model_optimization.ipynb) | does cleaning help? · hyperparameter search | flex |
| 10 | [30 pandas tricks (+ LLM copilot)](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/10_bonus_pandas_tricks.ipynb) | a reference you'll reuse for years | flex |
| 11 | [🧪 Reinforcement learning](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/11_reinforcement_learning_qlearning.ipynb) | learn a sepsis treatment policy, AI-Clinician style | flex |
| 12 | [📦 When data is too big for pandas](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/12_big_data_beyond_pandas.ipynb) | Parquet, chunking, Polars/DuckDB/Dask/Spark | flex |
| 13 | [⚖️ Fairness & subgroup performance](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/13_fairness_and_subgroups.ipynb) | does the model work equally well for everyone? | flex |
| 14 | [🏆 Capstone challenge](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/14_capstone_challenge.ipynb) | beat the baseline on a held-out set | flex |

**⭐ Notebook 04 is the heart of the course.** Clinical data is *temporal*; turning a timeline into
features is what separates a toy model from a useful one.

Notebooks marked **flex** are yours to keep and work through at your own pace — some may be covered
live if the day runs ahead. The full schedule is just below.

---

## ⏱️ How the 6 hours run

| Time | Session |
|------|---------|
| 00:00 – 00:20 | Welcome · why ML in medicine · the dataset · how Colab works · **this notebook** |
| 00:20 – 01:15 | **01 Pandas for clinical data** |
| 01:15 – 02:10 | **02 Data cleaning** |
| 02:10 – 02:25 | ☕ Break |
| 02:25 – 03:00 | **03 Exploratory data analysis** |
| 03:00 – 04:10 | ⭐ **04 Time-series feature engineering** |
| 04:10 – 04:40 | 🍽️ Lunch / long break |
| 04:40 – 05:45 | **05 First predictive models** |
| 05:45 – 06:30 | **06 Evaluating like a clinician** |
| 06:30 – 07:05 | **07 SHAP** · **08 Pitfalls & leakage** (highlights) |
| — flex — | 09 · 10 · 11 RL · 12 big data · 13 fairness · 🏆 14 capstone |

*(Times are offsets from the start — a 09:00 start ends around 16:05 with breaks.)*

If you ever fall behind, the irreducible core is **01 → 02 → 04 → 05 → 06**. Every notebook reads
fine on its own, so nothing is lost by finishing one at home.

---

## 🩹 If something breaks

| Symptom | Fix |
|---------|-----|
| `FileNotFoundError` / "Upload …" | The CSV isn't in this session. Re-run the setup cell and upload from `data/`. |
| Colab session reset / "runtime disconnected" | Re-run the setup cell. With Drive connected the data reloads by itself. |
| `Installing: shap xgboost` hangs a moment | Normal on Colab (≈20 s), only the first time. |
| A plot looks different from your neighbour's | Fine — small dataset, random seeds. Numbers should be in the same ballpark. |
| You get ROC-AUC ≈ 1.00 | You've almost certainly leaked. That's Notebook 08 — celebrate it and go look. |

## ⚖️ Ground rules for the data

- It is **real, de-identified patient data**. Do not attempt to re-identify anyone.
- **Do not redistribute the CSVs** — a data use agreement covers them. Share the *notebooks* freely.
- **Never paste patient rows into ChatGPT or any public LLM.** Schema and `describe()` summaries are
  usually fine; individual records are not. Notebook 10 has the safe recipes.
- Nothing built today is validated for clinical use.

Have fun — and by the end of the day, open one of *your* CSVs and try the same recipes. 🚀